# Validação 27 — múltiplas alegações por artigo

## Objetivo

Validar, de forma controlada e reproduzível, que um artigo pode originar várias alegações atômicas, sem duplicatas, e que cada uma recebe sua própria consulta e seu próprio resultado.

## Configuração

O cenário abaixo é sintético e inspirado no exemplo da rodopsina. Ele valida o contrato de software, não a correção científica das alegações. A segunda alegação é uma duplicata proposital da primeira.

In [1]:
import json
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from fatofake.article_ingestion import (
    ArticleFirstAnalysisRunner, ArticleSubmission, GeminiArticleExtractor,
    ResolvedArticleDocument,
)
from fatofake.document_parsing import ParsedPage

## Alegações candidatas

In [2]:
claims = [
    {'text': 'Hydrostatic pressure changes rhodopsin hydration.', 'quote': 'Hydrostatic pressure changed rhodopsin hydration.', 'search_query': 'hydrostatic pressure rhodopsin hydration'},
    {'text': 'Hydrostatic pressure changes rhodopsin hydration!', 'quote': 'Hydrostatic pressure changed rhodopsin hydration.', 'search_query': 'rhodopsin hydration hydrostatic pressure'},
    {'text': 'Osmotic pressure affects rhodopsin activation.', 'quote': 'Osmotic pressure affected rhodopsin activation.', 'search_query': 'osmotic pressure rhodopsin activation'},
    {'text': 'Structural and solvent water have different roles in rhodopsin.', 'quote': 'Structural water and solvent water had distinct roles.', 'search_query': 'structural solvent water rhodopsin roles'},
    {'text': 'Hydration changes alter rhodopsin conformational equilibrium.', 'quote': 'Hydration changes shifted the conformational equilibrium.', 'search_query': 'hydration rhodopsin conformational equilibrium'},
]
document_text = ' '.join(item['quote'] for item in claims)

class ControlledGateway:
    model_name = 'controlled-gemini'
    def _post_json(self, _url, _payload):
        output = {
            'title': 'Water and rhodopsin', 'doi': '10.1000/rhodopsin',
            'primary_claim': claims[0]['text'],
            'primary_claim_quote': claims[0]['quote'],
            'search_query': claims[0]['search_query'],
            'research_context': 'BASIC_SCIENCE',
            'additional_claims': [item['text'] for item in claims[1:]],
            'absolute_language': [], 'claims': claims,
        }
        return {'candidates': [{'content': {'parts': [{'text': json.dumps(output)}]}}]}
    @staticmethod
    def _response_text(payload):
        return payload['candidates'][0]['content']['parts'][0]['text']

submission = ArticleSubmission('https://example.org/rhodopsin', 'url', None, None, None)
resolved = ResolvedArticleDocument(
    title='Water and rhodopsin', doi='10.1000/rhodopsin', text=document_text,
    pmid='12345678', parser_name='controlled-full-text', page_count=1,
    pages=(ParsedPage(6, document_text),), sections=(('Results', document_text),),
    content_scope='FULL_TEXT',
)

## Extração e deduplicação

In [3]:
extractor = GeminiArticleExtractor(ControlledGateway())
extracted = extractor.extract(submission, resolved)
rows = [
    {
        'id': item.claim_id, 'alegação': item.text, 'consulta': item.search_query,
        'seção': item.section, 'página': item.page,
    }
    for item in extracted.claims
]
rows

[{'id': 'claim-01',
  'alegação': 'Hydrostatic pressure changes rhodopsin hydration.',
  'consulta': 'hydrostatic pressure rhodopsin hydration',
  'seção': 'Results',
  'página': 6},
 {'id': 'claim-02',
  'alegação': 'Osmotic pressure affects rhodopsin activation.',
  'consulta': 'osmotic pressure rhodopsin activation',
  'seção': 'Results',
  'página': 6},
 {'id': 'claim-03',
  'alegação': 'Structural and solvent water have different roles in rhodopsin.',
  'consulta': 'structural solvent water rhodopsin roles',
  'seção': 'Results',
  'página': 6},
 {'id': 'claim-04',
  'alegação': 'Hydration changes alter rhodopsin conformational equilibrium.',
  'consulta': 'hydration rhodopsin conformational equilibrium',
  'seção': 'Results',
  'página': 6}]

Foram fornecidos cinco candidatos, mas a variação apenas por pontuação da primeira alegação foi removida. Permaneceram quatro perguntas científicas distintas e rastreáveis.

## Análises independentes

In [4]:
class RecordingEvidenceRunner:
    def __init__(self): self.calls = []
    def analyze(self, claim, article_reference=None, *, excluded_dois=(), query_override=None, related_seed_pmids=()):
        self.calls.append({'claim': claim, 'query': query_override})
        return {'verification': {'alerts': []}, 'articles': []}

class ControlledResolver:
    def resolve(self, _submission): return resolved

evidence_runner = RecordingEvidenceRunner()
runner = ArticleFirstAnalysisRunner(extractor, evidence_runner, reference_resolver=ControlledResolver())
result = runner.analyze_article(submission)
[
    {
        'id': item['claim_id'],
        'alegação': item['claim']['text'],
        'consulta executada': evidence_runner.calls[index]['query'],
        'resultado': item['result']['user_summary']['headline'],
    }
    for index, item in enumerate(result['claim_analyses'])
]

[{'id': 'claim-01',
  'alegação': 'Hydrostatic pressure changes rhodopsin hydration.',
  'consulta executada': 'hydrostatic pressure rhodopsin hydration',
  'resultado': 'Ainda não há evidência direta suficiente para comparar'},
 {'id': 'claim-02',
  'alegação': 'Osmotic pressure affects rhodopsin activation.',
  'consulta executada': 'osmotic pressure rhodopsin activation',
  'resultado': 'Ainda não há evidência direta suficiente para comparar'},
 {'id': 'claim-03',
  'alegação': 'Structural and solvent water have different roles in rhodopsin.',
  'consulta executada': 'structural solvent water rhodopsin roles',
  'resultado': 'Ainda não há evidência direta suficiente para comparar'},
 {'id': 'claim-04',
  'alegação': 'Hydration changes alter rhodopsin conformational equilibrium.',
  'consulta executada': 'hydration rhodopsin conformational equilibrium',
  'resultado': 'Ainda não há evidência direta suficiente para comparar'}]

## Verificações

In [5]:
assert len(claims) == 5
assert len(extracted.claims) == 4
assert len(result['claim_analyses']) == 4
assert len(evidence_runner.calls) == 4
assert all(item.section == 'Results' and item.page == 6 for item in extracted.claims)
assert all(
    analysis['claim']['text'] == analysis['result']['user_summary']['claim']
    for analysis in result['claim_analyses']
)
print('OK: 5 candidatos → 4 alegações únicas → 4 buscas e 4 resultados independentes.')

OK: 5 candidatos → 4 alegações únicas → 4 buscas e 4 resultados independentes.


## Próximos passos

A etapa comprova o isolamento entre alegações, suas consultas e seus resultados. Ela não avalia a precisão científica do modelo. A próxima etapa substituirá o indicador percentual único por cobertura, balanço das evidências e confiança metodológica separados.